In [171]:
import requests
import pandas as pd
from io import StringIO
import time

In [3]:
locations = [
    "Paro", "Gasa", "Deothang", "Bhur", "Nganglam", "Zhemgang", "Simtokha",
    "Phuentsholing", "Chamkhar", "Kanglung_Tashigang", "Mongar", "Trongsa",
    "Tashiyangtse", "Tangmachu_Lhuentse"
]

In [4]:
latitudes = [
    27.43,  # Paro
    27.91,  # Gasa
    26.88,  # Deothang
    26.85,  # Bhur
    27.05,  # Nganglam
    27.21,  # Zhemgang
    27.46,  # Simtokha
    26.86,  # Phuentsholing
    27.56,  # Chamkhar
    27.26,  # Kanglung (Trashigang)
    27.27,  # Mongar
    27.50,  # Trongsa
    27.63,  # Tashiyangtse
    27.77   # Tangmachu (Lhuentse)
]

longitudes = [
    89.42,  # Paro
    89.73,  # Gasa
    91.56,  # Deothang
    90.56,  # Bhur
    91.20,  # Nganglam
    90.67,  # Zhemgang
    89.64,  # Simtokha
    89.39,  # Phuentsholing
    90.73,  # Chamkhar
    91.23,  # Kanglung (Trashigang)
    91.25,  # Mongar
    90.50,  # Trongsa
    91.51,  # Tashiyangtse
    91.01   # Tangmachu (Lhuentse)
]


In [127]:
# --- Only the parameters you requested ---
parameters = [
    "PRECTOTCORR",  # Rainfall
    "T2M",          # Temperature at 2m
    "CLOUD_AMT",    # Cloud cover
    "RH2M",         # Humidity
    "ALLSKY_SFC_SW_DWN"  # Solar radiation
]

In [174]:
class FetchNASAAPIData:
    def __init__(self, latitude, longitude, parameters):
        self.latitude=latitude
        self.longitude=longitude
        self.parameters=parameters
        self.df = pd.DataFrame()
        self.dataLength=0

    def generateAPIURL(self, startDate, endDate, latitude, longitude, dataFormat="JSON"):
        url = (
                f"https://power.larc.nasa.gov/api/temporal/daily/point?"
                f"parameters={','.join(self.parameters)}&start={startDate}&end={endDate}"
                f"&latitude={latitude}&longitude={longitude}&community=AG&format={dataFormat}"
            )
        return url

    def fetchAPI(self, startDate, endDate, latitude, longitude, dataFormat="JSON"):
        try:
            response = requests.get(self.generateAPIURL(startDate=startDate, endDate=endDate, latitude=latitude, longitude=longitude))
            response.raise_for_status()  # Raises HTTPError if status != 200
            data = response.json()
        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred: {http_err}. Retrying in 60 seconds...")
            time.sleep(60)
            try:
                response = requests.get(self.generateAPIURL(startDate=startDate, endDate=endDate, latitude=latitude, longitude=longitude))
                response.raise_for_status()
                data = response.json()
            except requests.exceptions.HTTPError as err:
                print(f"Failed again: {err}")
            
        except requests.exceptions.RequestException as err:
            print(f"Other request error occurred: {err}")  # e.g. connection issues
        except ValueError as json_err:
            print(f"Error decoding JSON: {json_err}")
        else:
            print("Request successful.")
            return data

    def generateDictionary(self, APIdata, columns):
        data=APIdata
        self.dataLength=len(data["properties"]["parameter"][self.parameters[0]].values())
        generatedDictionary = {
                "date": data["properties"]["parameter"]["PRECTOTCORR"].keys(),
                "longitude": self.dataLength*[data["geometry"]["coordinates"][0]],
                "latitude": self.dataLength*[data["geometry"]["coordinates"][1]],
                "Elevation": self.dataLength*[data["geometry"]["coordinates"][2]],
            }
        constantParameters = ["date", "longitude", "latitude", "Elevation"]
        reducedColumns = [x for x in columns if x not in constantParameters]
        for parameter, column in zip(self.parameters, reducedColumns):
            generatedDictionary[column] = data["properties"]["parameter"][parameter].values()

        return generatedDictionary

    def appendDf(self, dectionary):
        self.df = pd.concat([self.df, pd.DataFrame(dectionary)], ignore_index=True)
        return self.df

    def maintainServerError(self, i):
        backupDictionary = {
            "i": i,
            "df": self.df
        }

    def fetchDataAllDistricts(self, latitudes, longitudes, locations, columns, startDate, endDate):
        for i in range(len(latitudes)):
            data = self.fetchAPI(startDate, endDate, latitudes[i], longitudes[i])
            dictDistrict = self.generateDictionary(data, columns)
            self.appendDf(dictDistrict)

In [175]:
columns = [
            "date",
            "longitude",
            "latitude",
            "Rain",
            "T2M",            
            "CLOUD_AMT",       
            "RH2M",          
            "ALLSKY_SFC_SW_DWN",
            "Elevation",
        ]
fetchNASA = FetchNASAAPIData(latitudes[0], longitudes[0], parameters)
fetchNASA.fetchDataAllDistricts(latitudes, longitudes, locations, columns, startDate, endDate)

Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
Request successful.
HTTP error occurred: 502 Server Error: Bad Gateway for url: https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR,T2M,CLOUD_AMT,RH2M,ALLSKY_SFC_SW_DWN&start=20010101&end=20250601&latitude=27.63&longitude=91.51&community=AG&format=JSON. Retrying in 60 seconds...


TypeError: 'NoneType' object is not subscriptable

In [177]:
fetchNASA.df.tail()

,date,longitude,latitude,Elevation,Rain,T2M,CLOUD_AMT,RH2M,ALLSKY_SFC_SW_DWN
107011,20250528,90.5,27.5,3134.16,7.61,13.78,-999.0,80.18,17.92
107012,20250529,90.5,27.5,3134.16,14.99,12.30,-999.0,84.36,17.02
107013,20250530,90.5,27.5,3134.16,29.19,12.72,-999.0,87.69,10.97
107014,20250531,90.5,27.5,3134.16,11.11,13.78,-999.0,86.85,14.78
107015,20250601,90.5,27.5,3134.16,29.01,13.48,-999.0,87.96,18.64


In [179]:
fetchNASA.df.to_csv("output.csv", index=False)